# ♟️ 0x960 — GRPO x OpenEnv Training Notebook

A clean, end-to-end Colab workflow for training an agent policy over a bounded Chess960 engine-editing environment.

**what this runs**
- OpenEnv server startup + health check
- environment sanity passes (handcrafted + inference)
- optional teacher/distill and student SFT stages
- TRL GRPO refinement on structured actions (`write_file`, `run_match`, `finish`)
- artifact + benchmark inspection

**links**
- repo: <https://github.com/qtzx06/0x960>
- training notes: <https://github.com/qtzx06/0x960/blob/main/docs/training.md>


## quick run guide
1. run cells top-to-bottom once
2. if GPU is available, keep it enabled for train mode
3. use the default commands first, then tune `--steps` / `--num-generations`
4. check final artifact cells before ending runtime


## 0) runtime + dependencies
Use Colab GPU if available.


In [ ]:
%%capture
!pip -q install uv
!uv pip install --system "openenv-core[core]>=0.2.1" "trl>=0.19.0" "transformers>=4.57.0" "accelerate>=0.34.0" "datasets>=3.0.0"
print('deps installed')


In [ ]:
!nvidia-smi || true
!python -V


## 1) clone + install project


In [ ]:
!git clone https://github.com/qtzx06/0x960.git
%cd 0x960
!uv pip install --system -e .[openenv,train]


## 2) start local OpenEnv server


In [ ]:
import subprocess, time
server = subprocess.Popen(['python','-m','uvicorn','zero960_env.server.app:app','--host','127.0.0.1','--port','8000'])
time.sleep(3)
print('openenv server pid:', server.pid)
!curl -s http://127.0.0.1:8000/health || true


## 3) environment sanity checks
These verify the exact environment/action contract before training.


In [ ]:
!python -m train.minimal_trl_openenv --mode handcrafted --base-url http://127.0.0.1:8000


In [ ]:
# optional quick inference check
!python -m train.minimal_trl_openenv --mode infer --base-url http://127.0.0.1:8000 --model Qwen/Qwen3.5-0.8B


## 4) teacher trajectory collection (optional)
Optional but recommended: bootstrap good action priors before GRPO.


In [ ]:
# requires teacher model/API setup used in your run environment
# !python -m train.codex_distill --base-url http://127.0.0.1:8000 --model gpt-5.4 --episodes 20
!ls -lah outputs/codex_distill 2>/dev/null || true


## 5) student SFT (optional, recommended)


In [ ]:
# !python -m train.sft_student --model Qwen/Qwen3.5-0.8B --output-dir outputs/sft_qwen_0p8b
!ls -lah outputs/sft_qwen_0p8b 2>/dev/null || true


## 6) GRPO training (core loop)
This is the primary RL stage over bounded actions and environment-grounded rewards.


In [ ]:
!python -m train.minimal_trl_openenv --mode train --base-url http://127.0.0.1:8000 --model Qwen/Qwen3.5-0.8B --steps 20 --num-generations 4 --output-dir outputs/grpo_qwen_0p8b


## 7) inspect artifacts + observability logs


In [ ]:
!find outputs -maxdepth 3 -type f | head -200


In [ ]:
# benchmark candidate vs baseline (optional)
!python -m train.benchmark_eval --candidate-file src/zero960/workspace_template/eval.py --baseline-file src/zero960/engine/default_eval.py --positions 32 --depth 2


## 8) expected outcomes
- environment-grounded RL over multi-step engineering actions
- bounded tool-use policy optimization (not plain completion tuning)
- teacher/SFT bootstrap + GRPO refinement path used in 0x960
